In [3]:
pip install transformers torch torchvision pillow requests accelerate

In [7]:
# initialise the model and the processor

import torch
import requests
from PIL import Image
from transformers import AutoProcessor, Florence2ForConditionalGeneration

# loading the model
model_id = "florence-community/Florence-2-base"

model = Florence2ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)
processor  = AutoProcessor.from_pretrained(model_id)

#loading a sample of an image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

task_prompt = "<OD>"

#prepare inputs for the model
inputs = processor(text=task_prompt, 
                   images=image, 
                   return_tensors="pt"
                   ).to(model.device, torch.bfloat16)

#generate the output
generated_ids = model.generate(
    **inputs,
    max_new_tokens=1024,
    num_beams=3,
)
#decode the raw text
generated_text = processor.batch_decode(generated_ids, 
                                        skip_special_tokens=False
                                        )[0]

#Parse the text into structured coordinates
image_size = image.size
parsed_answer = processor.post_process_generation(generated_text, 
                                                  task=task_prompt, 
                                                  image_size=image_size)

print(parsed_answer)


Loading weights:   0%|          | 0/665 [00:00<?, ?it/s]

{'<OD>': {'bboxes': [[405, 271, 439, 286], [328, 252, 345, 276], [380, 270, 405, 281], [480, 280, 502, 293], [499, 283, 514, 296], [0, 285, 196, 374], [190, 290, 291, 354], [329, 301, 393, 335], [532, 301, 558, 314], [134, 98, 175, 145], [71, 169, 92, 290]], 'labels': ['awning', 'awning', 'awning', 'awning', 'awning', 'car', 'car', 'car', 'car', 'signboard', 'street light']}}
